# Exercice 2 - Entraînez un agent avec une table de décision (Q-table)

Dans cet exercice, on implémente de zéro le premier vrai algorithme d'apprentissage par renforcement : le **Q-Learning**.

L'idée est de construire un tableau qui note, pour chaque situation, la qualité de chaque action possible. C'est la **Q-table**.

Trois étapes :

1. Créer l'environnement `FrozenLake-v1` et initialiser la Q-table à zéro.
2. Coder la boucle d'entraînement qui remplit cette table.
3. Évaluer l'agent entraîné et mesurer son taux de réussite.

In [1]:
# Installation des librairies
%pip install -q gymnasium numpy

Note: you may need to restart the kernel to use updated packages.


C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P11\.venv\Scripts\python.exe: No module named pip


In [2]:
import gymnasium as gym
import numpy as np
import random

print("Version de Gymnasium :", gym.__version__)
print("Version de NumPy     :", np.__version__)

Version de Gymnasium : 1.3.0
Version de NumPy     : 2.5.3


## Étape 1 : préparer l'environnement et la Q-table

### FrozenLake, c'est quoi ?

Un lac gelé de 4 cases sur 4. On part de la case en haut à gauche et on doit atteindre le cadeau en bas à droite. Certaines cases sont des trous : si on tombe dedans, la partie est finie et c'est perdu.

La récompense vaut **1 uniquement si on atteint le cadeau**, et 0 dans tous les autres cas. Tomber dans un trou rapporte 0, faire un step rapporte 0.

### Le piège : la glace glisse

C'est le comportement par défaut de `FrozenLake-v1`, et ça change beaucoup de choses. Quand on décide d'aller vers le bas, on ne va vers le bas qu'**une fois sur trois**. Les deux autres fois, on dérape à gauche ou à droite.

Autrement dit l'agent ne contrôle pas complètement ce qui lui arrive. 

### Pourquoi FrozenLake se prête bien au Q-Learning

Parce que FrozenLake n'a que **16 situations possibles**, les 16 cases du lac. On peut donc toutes les lister dans un tableau. Un environnement dont les situations se décrivent par des nombres à virgule en aurait une infinité, et le tableau serait impossible à construire.

In [3]:
env = gym.make("FrozenLake-v1")

# La carte du lac
print("La carte du lac :")
for ligne in env.unwrapped.desc:
    print("   ", " ".join(case.decode() for case in ligne))
print("    S = départ, F = glace, H = trou, G = cadeau")
print()

# Les dimensions dont on a besoin pour construire la Q-table
n_etats = env.observation_space.n
n_actions = env.action_space.n
print("Nombre d'états   :", n_etats)
print("Nombre d'actions :", n_actions)
print()

# La Q-table, remplie de zéros
q_table = np.zeros((n_etats, n_actions))
print("Dimensions de la Q-table :", q_table.shape)
print(q_table)

La carte du lac :
    S F F F
    F H F H
    F F F H
    H F F G
    S = départ, F = glace, H = trou, G = cadeau

Nombre d'états   : 16
Nombre d'actions : 4

Dimensions de la Q-table : (16, 4)
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


La Q-table fait bien **(16, 4)**, comme demandé dans l'énoncé.

* **16 lignes**, une par case du lac. Les cases sont numérotées de 0 à 15, de gauche à droite puis de haut en bas.
* **4 colonnes**, une par action possible : `0` gauche, `1` bas, `2` droite, `3` haut.

Chaque case du tableau contient une note qui répond à la question : *si on est sur cette case et qu'on fait cette action, à quel point est-ce une bonne idée ?*

Pour l'instant tout vaut zéro, ce qui veut dire que **l'agent ne sait absolument rien**. Le remplir, c'est tout le travail de l'étape 2.

## Étape 2 : implémenter la boucle de Q-Learning

C'est le cœur de l'exercice. À chaque step, l'agent doit choisir entre deux comportements.

### Explorer ou exploiter

On tire un nombre au hasard entre 0 et 1 :

* s'il est **inférieur à `epsilon`**, l'agent **explore** : il joue une action au hasard
* sinon, il **exploite** : il joue la meilleure action qu'il connaît, avec `np.argmax(q_table[state, :])`

`epsilon` démarre à 1.0, donc au début l'agent explore tout le temps, ce qui est logique puisqu'il ne sait rien. Puis on le **réduit à la fin de chaque épisode** : plus l'agent apprend, moins il a besoin de tâtonner.

### La mise à jour de la Q-table

Après chaque action, on corrige la note de la paire (état, action) qu'on vient d'utiliser :

```
nouvelle_valeur = ancienne_valeur + lr * (recompense + gamma * max_q_futur - ancienne_valeur)
```

Ce qu'il y a entre les parenthèses, c'est l'écart entre **ce que l'agent croyait** et **ce qu'il vient de constater**. Le `lr` (learning rate) décide de quelle part de cet écart on corrige : trop grand et l'agent change d'avis à chaque fois, trop petit et il apprend trop lentement.

Attention au point de vigilance de l'énoncé : le `max_q_futur` se calcule sur `new_state`, la case où l'agent vient d'arriver, pas sur celle d'où il vient.

In [4]:
# Les hyperparamètres de l'apprentissage
n_episodes = 10000        # nombre d'épisodes d'entraînement
learning_rate = 0.1       # à quelle vitesse on corrige la Q-table
gamma = 0.99              # importance donnée aux récompenses futures

epsilon = 1.0             # au départ l'agent explore tout le temps
epsilon_min = 0.01        # mais il gardera toujours un peu de hasard
epsilon_decay = 0.9995    # de combien epsilon diminue après chaque épisode

In [5]:
q_table = np.zeros((n_etats, n_actions))   # on repart d'une table vierge

for episode in range(n_episodes):

    state, info = env.reset()
    terminated = False
    truncated = False

    while not terminated and not truncated:

        # Explorer ou exploiter ?
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()        # exploration
        else:
            action = np.argmax(q_table[state, :])     # exploitation

        new_state, reward, terminated, truncated, info = env.step(action)

        # Mise à jour de la Q-table sur la paire (state, action) qui vient d'être utilisée
        ancienne_valeur = q_table[state, action]
        max_q_futur = np.max(q_table[new_state, :])
        q_table[state, action] = ancienne_valeur + learning_rate * (
            reward + gamma * max_q_futur - ancienne_valeur
        )

        state = new_state

    # On réduit epsilon à la fin de chaque épisode
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("Entraînement terminé sur", n_episodes, "épisodes.")
print("Epsilon final :", round(epsilon, 4))

Entraînement terminé sur 10000 épisodes.
Epsilon final : 0.01


In [6]:
print("La Q-table après entraînement :")
print(np.round(q_table, 3))

La Q-table après entraînement :
[[0.54  0.489 0.509 0.497]
 [0.303 0.381 0.286 0.509]
 [0.403 0.417 0.426 0.472]
 [0.387 0.271 0.338 0.452]
 [0.558 0.404 0.229 0.392]
 [0.    0.    0.    0.   ]
 [0.303 0.162 0.201 0.138]
 [0.    0.    0.    0.   ]
 [0.429 0.324 0.394 0.601]
 [0.438 0.672 0.499 0.505]
 [0.68  0.258 0.415 0.383]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.508 0.509 0.766 0.504]
 [0.71  0.888 0.761 0.77 ]
 [0.    0.    0.    0.   ]]


Le tableau n'est plus rempli de zéros, l'agent a appris quelque chose. Trois choses se lisent dessus :

* Les lignes qui **restent à zéro** correspondent aux trous et à la case d'arrivée. Une fois dessus la partie est finie, donc l'agent n'y prend jamais de décision et ces lignes ne sont jamais mises à jour.
* Les valeurs sont **plus grandes près de l'arrivée**, tout simplement parce que la récompense y est plus proche.
* Sur chaque ligne, **la plus grande valeur indique l'action que l'agent choisira** à cet endroit.

## Étape 3 : évaluer l'agent entraîné

On refait tourner des épisodes, mais avec deux différences importantes :

* **plus aucune exploration**. L'agent joue systématiquement `np.argmax(q_table[state, :])`, il n'y a plus d'epsilon.
* **plus aucune mise à jour de la Q-table**. L'apprentissage est terminé, on ne fait plus que mesurer.

On compte une victoire à chaque fois qu'un épisode se termine avec une récompense de 1.0, c'est-à-dire quand l'agent a atteint le cadeau.

In [7]:
n_episodes_eval = 100
total_wins = 0

for episode in range(n_episodes_eval):

    state, info = env.reset()
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = np.argmax(q_table[state, :])   # toujours la meilleure action connue
        state, reward, terminated, truncated, info = env.step(action)

    if reward == 1.0:
        total_wins += 1

env.close()

taux_de_reussite = 100 * total_wins / n_episodes_eval
print(f"Taux de réussite sur {n_episodes_eval} épisodes : {taux_de_reussite:.0f}%")

Taux de réussite sur 100 épisodes : 74%


## Ce qu'on observe

L'agent atteint maintenant le cadeau dans la majorité des cas, alors qu'au départ sa Q-table était entièrement vide. Il a appris tout seul, uniquement en jouant et en regardant les récompenses qu'il recevait.

Il ne fait pas 100 %, et c'est normal : **la glace glisse**. Même en jouant la meilleure action connue à chaque step, l'agent dérape deux fois sur trois et finit parfois dans un trou sans l'avoir voulu. Le plafond de cet environnement se situe aux alentours des trois quarts de réussite.

Une précaution de lecture importante : **l'évaluation ne porte que sur 100 épisodes**, ce qui est peu. Avec exactement la même Q-table, le chiffre affiché peut aussi bien tomber sur 65 % que sur 85 % selon la chance du tirage. Pour une mesure fiable il faudrait évaluer sur beaucoup plus d'épisodes.

À cela s'ajoute que l'entraînement lui-même contient du hasard, donc deux entraînements ne produisent pas exactement la même Q-table.